# AeroGPT Pre-training

Pre-train the trajectory foundation model on historical ADS-B data from OpenSky Network.

**Runtime**: GPU (A100 recommended). **Estimated time**: 8-12 hours for 100K steps.

In [ ]:
# Cell 1: Setup
import subprocess, sys, os

# Install dependencies
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torch-geometric', 'polars', 'pyarrow', 'pyopensky',
    'structlog', 'pyyaml', 'scipy', 'matplotlib', 'tqdm', 'rich', 'httpx'])

# Clone repo and install package
if os.path.exists('/content'):
    os.chdir('/content')
    if not os.path.exists('/content/aeroconform'):
        subprocess.check_call(['git', 'clone', 'https://github.com/Nadosaurusrex/aeroconform.git'])
    os.chdir('/content/aeroconform')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    # Ensure src is importable even if editable install has issues
    if '/content/aeroconform' not in sys.path:
        sys.path.insert(0, '/content/aeroconform')

# Configure pyopensky credentials
conf_dir = os.path.expanduser('~/.config/pyopensky')
os.makedirs(conf_dir, exist_ok=True)
with open(os.path.join(conf_dir, 'settings.conf'), 'w') as f:
    f.write('[default]\nusername = nadosaurusrex\npassword = Pescespada28\n')
    f.write('client_id = nadosaurusrex-api-client\nclient_secret = 82HnTrXV17fwD3XJzPfJbFFHDhR0UPNi\n')
    f.write('\n[cache]\npurge = 90 days\n')

# Verify imports work
from src.data.opensky_client import TrinoClient
from src.models.aerogpt import AeroGPT
print('Setup complete - all imports verified')

In [ ]:
# Cell 2: Mount Google Drive for persistent storage
import os
from pathlib import Path

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/aeroconform/data')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/aeroconform/checkpoints')
else:
    DATA_DIR = Path('data')
    CHECKPOINT_DIR = Path('checkpoints')

DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data dir: {DATA_DIR}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

In [ ]:
# Cell 3: Download data via Trino
from src.data.opensky_client import TrinoClient, TrinoUnavailableError
from src.data.preprocessing import encode_state_vectors, delta_encode, compute_norm_stats
from src.data.flight_segmentation import segment_flights
from src.data.schemas import NormStats
from src.utils.logging import setup_logging
import polars as pl
from datetime import datetime, timedelta

setup_logging(level='INFO')

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Query 1 week of LIMM FIR data (sufficient for pre-training)
START = '2024-08-01'
END = '2024-08-08'

try:
    client = TrinoClient()
    start_dt = datetime.fromisoformat(START)
    end_dt = datetime.fromisoformat(END)
    current = start_dt
    
    while current < end_dt:
        chunk_end = min(current + timedelta(days=1), end_dt)
        date_str = current.strftime('%Y-%m-%d')
        output_path = RAW_DIR / f'state_vectors_{date_str}.parquet'
        
        if not output_path.exists():
            print(f'Downloading {date_str}...')
            df = client.query_state_vectors(current, chunk_end)
            if len(df) > 0:
                encoded = encode_state_vectors(df)
                encoded.write_parquet(output_path)
                print(f'  Saved {len(encoded)} rows')
        else:
            print(f'  {date_str} already exists')
        current = chunk_end
    
    print('Data download complete')
except TrinoUnavailableError as e:
    print(f'Trino unavailable: {e}')
    print('Please ensure Trino access is configured. See CLAUDE.md for setup.')

In [ ]:
# Cell 4: Compute normalization stats and prepare dataset
import numpy as np
from pathlib import Path

parquet_files = sorted(RAW_DIR.glob('*.parquet'))
print(f'Found {len(parquet_files)} Parquet files')

# Segment flights and compute norm stats
all_flights = []
all_deltas = []

for pf in parquet_files:
    df = pl.read_parquet(pf)
    flights = segment_flights(df)
    all_flights.extend(flights)
    for flight in flights:
        deltas = delta_encode(flight.features)
        all_deltas.append(deltas)

print(f'Total flights: {len(all_flights)}')

# Compute and save norm stats
norm_stats = compute_norm_stats(all_deltas)
stats_path = str(DATA_DIR / 'norm_stats.npz')
norm_stats.save(stats_path)
print(f'Mean: {norm_stats.mean}')
print(f'Std: {norm_stats.std}')

In [ ]:
# Cell 5: Create datasets and dataloaders
import torch
from torch.utils.data import DataLoader, random_split
from src.data.dataset import TrajectoryMapDataset, collate_trajectories
from src.data.schemas import NormStats

norm_stats = NormStats.load(str(DATA_DIR / 'norm_stats.npz'))

dataset = TrajectoryMapDataset(all_flights, context_length=128, norm_stats=norm_stats)
print(f'Total windows: {len(dataset)}')

# Train/val split (90/10)
val_size = max(1, len(dataset) // 10)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset, batch_size=256, shuffle=True,
    collate_fn=collate_trajectories, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False,
    collate_fn=collate_trajectories, num_workers=2, pin_memory=True
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Cell 6: Initialize model and trainer
from src.models.aerogpt import AeroGPT
from src.training.trainer import Trainer
from src.utils.config import ModelConfig, TrainingConfig

model_config = ModelConfig.from_yaml(Path('configs/model.yaml'))
train_config = TrainingConfig.from_yaml(Path('configs/train.yaml'))

model = AeroGPT(model_config)
print(f'Model parameters: {model.count_parameters():,}')

trainer = Trainer(
    model=model,
    config=train_config,
    train_loader=train_loader,
    val_loader=val_loader,
    checkpoint_dir=CHECKPOINT_DIR,
)
print(f'Device: {trainer.device}')
print(f'AMP enabled: {trainer.use_amp}')

In [ ]:
# Cell 7: Quick sanity check (100 steps)
import copy

quick_config = copy.copy(train_config)
quick_config.max_steps = 100
quick_config.eval_every = 50
quick_config.checkpoint_every = 100000  # Don't checkpoint

quick_trainer = Trainer(
    model=AeroGPT(model_config),
    config=quick_config,
    train_loader=train_loader,
    val_loader=val_loader,
)
history = quick_trainer.train()
print(f'Final loss: {history["train_loss"][-1]:.4f}')
print('Sanity check passed!')

In [ ]:
# Cell 8: Full pre-training (100K steps)
# Resume from checkpoint if available
resume_path = CHECKPOINT_DIR / 'best.pt'
if resume_path.exists():
    print('Resuming from checkpoint...')
    trainer.load_checkpoint(resume_path)

history = trainer.train()
print(f'Pre-training complete. Steps: {trainer.global_step}')

In [ ]:
# Cell 9: Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('NLL')
axes[0].set_yscale('log')

if history['val_loss']:
    eval_steps = list(range(train_config.eval_every, train_config.max_steps + 1, train_config.eval_every))
    axes[1].plot(eval_steps[:len(history['val_loss'])], history['val_loss'])
    axes[1].set_title('Validation Loss')
    axes[1].set_xlabel('Step')

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Step')

plt.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'pretrain_curves.png'), dpi=150)
plt.show()

In [ ]:
# Cell 10: Masked fine-tuning (20K steps)
from src.training.masked_trainer import MaskedTrainer
import copy

masked_config = copy.copy(train_config)
masked_config.max_steps = 20000
masked_config.warmup_steps = 500
masked_config.learning_rate = 1e-4  # Lower LR for fine-tuning

# Load best pre-trained checkpoint
best_ckpt = CHECKPOINT_DIR / 'best.pt'
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print('Loaded pre-trained checkpoint for masked fine-tuning')

masked_trainer = MaskedTrainer(
    model=model,
    config=masked_config,
    train_loader=train_loader,
    val_loader=val_loader,
    checkpoint_dir=CHECKPOINT_DIR,
    mask_ratio=0.15,
)

masked_history = masked_trainer.train()
print(f'Masked fine-tuning complete. Steps: {masked_trainer.global_step}')

# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': model_config,
    'norm_stats_path': str(DATA_DIR / 'norm_stats.npz'),
}, CHECKPOINT_DIR / 'aerogpt_final.pt')
print('Final model saved')